# V2 push — SFT v2 milestone exam (seed 3407, epoch 2)

**Runtime: L4 — MANDATORY** (frozen protocol). ~25–40 min, ~4–5 units.

One held-out run of the v2-push SFT (chosen checkpoint: **ep2**, by the A2
headroom rule applied to the distribution GRPO v2 will train on — new-dev
91.7 pass@8 vs ep1's 83.3). The 13b matrix showed v2 scoring +27 over v1 on
the exam-LIKE dev slice while v1-SFT actually scored BELOW BASE there — this
run tells us how much of that transfers to the real 164.

Context for the verdict: base 17.59 | SFT v1 s3407 23.63 | controlled-study
best mean 25.39 | target 30.4. No unsloth anywhere (peft merge, S2.22).
Optional second run of ep1 is in the RUNS list (skip-if-done) if units allow.

In [ ]:
# 1) GPU check
!nvidia-smi -L
import torch
assert torch.cuda.is_bf16_supported(), 'Switch runtime to L4.'
name = torch.cuda.get_device_name(0)
assert 'L4' in name, f'Protocol pins the exam to L4; this is {name}.'
print(name, '- OK')

In [ ]:
# 2) Drive + runs (ep2 primary; ep1 optional — delete from RUNS to skip)
from google.colab import drive
drive.mount('/content/drive')
import os, json, gc
PHASE8 = '/content/drive/MyDrive/rl-post-training/phase8'
RUNS = ['ep2']            # add 'ep1' for the optional second run
def met_path(ep):
    return f'{PHASE8}/metrics_sftv2_{ep}_s3407.json'
todo = [e for e in RUNS if not os.path.exists(met_path(e))]
print('to run:', todo)

In [ ]:
%%capture
!pip install -q peft
!pip uninstall -y -q torchao torchaudio torchvision timm   # strip BEFORE merging (peft rejects stale torchao)

In [ ]:
# 3) Merge v2 adapter(s) into full bf16 weights (pure peft — no unsloth)
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
BASE_ID = 'unsloth/Qwen2.5-Coder-1.5B-Instruct'
for ep in todo:
    out = f'/content/final_sftv2_{ep}'
    if os.path.exists(out):
        continue
    base = AutoModelForCausalLM.from_pretrained(BASE_ID, torch_dtype=torch.bfloat16)
    m = PeftModel.from_pretrained(base, f'{PHASE8}/sft_v2_s3407_{ep}')
    m = m.merge_and_unload()
    m.save_pretrained(out, safe_serialization=True)
    try:
        tok = AutoTokenizer.from_pretrained(f'{PHASE8}/sft_v2_s3407_{ep}')
    except Exception:
        tok = AutoTokenizer.from_pretrained(BASE_ID)
    tok.save_pretrained(out)
    del base, m; gc.collect(); torch.cuda.empty_cache()
    print('merged', ep, '->', out)
print('merges done')

In [ ]:
# 4) Harness at the frozen commit
FROZEN_COMMIT = '8fc5bae6479c4fbbb28c3f8b644f6a15b3f3b5bd'
%cd /content
!rm -rf bigcode-evaluation-harness
!git clone -q https://github.com/bigcode-project/bigcode-evaluation-harness.git
%cd /content/bigcode-evaluation-harness
!git checkout -q {FROZEN_COMMIT}
ver = !git rev-parse HEAD
assert ver[0] == FROZEN_COMMIT
!pip install -q -e .
!pip install -q -U transformers accelerate
!pip uninstall -y -q torchao torchaudio torchvision timm
import transformers
print('harness pinned', ver[0][:8], '| transformers', transformers.__version__)

In [ ]:
# 5) THE RUN(S) — frozen protocol
TASK = 'humanevalfixtests-python'
PROMPT = 'instruct'
BATCH = 16
%cd /content/bigcode-evaluation-harness
for ep in todo:
    print('=' * 70)
    print(f'SFT v2 {ep}: 164 x 20')
    gen_path = f'{PHASE8}/gens_sftv2_{ep}_s3407.json'
    m_path = met_path(ep)
    model_dir = f'/content/final_sftv2_{ep}'
    !accelerate launch main.py \
      --model {model_dir} --tasks {TASK} --prompt {PROMPT} \
      --do_sample True --temperature 0.2 --top_p 0.95 --n_samples 20 \
      --batch_size {BATCH} --precision bf16 \
      --max_length_generation 2048 --allow_code_execution \
      --save_generations --save_generations_path {gen_path} \
      --metric_output_path {m_path}
    print(open(m_path).read())

In [ ]:
# 6) VERDICT vs the whole ladder
def read_m(p):
    m = json.load(open(p)); k = [x for x in m if x != 'config'][0]
    return m[k].get('pass@1', 0) * 100, m[k].get('pass@10', 0) * 100
print(f"{'model':<22} pass@1    pass@10")
print(f"{'base':<22}  17.59%    23.50%")
print(f"{'SFT v1 s3407':<22}  23.63%    32.43%")
print(f"{'controlled best mean':<22}  25.39%    33.70%   (GRPO arm)")
for ep in ('ep1', 'ep2'):
    if os.path.exists(met_path(ep)):
        p1, p10 = read_m(met_path(ep))
        print(f"{'SFT v2 ' + ep + ' s3407':<22} {p1:6.2f}%   {p10:6.2f}%")
print(f"{'TARGET (OctoCoder-16B)':<22}  30.40%")
print('\nReading: v2 >= ~28 -> data lever transferred; GRPO v2 (+ random twin)')
print('gets a real shot at 30.4. v2 ~ 24-26 -> transfer partial; consider mixture')
print('rebalance or 3-seed v2 before RL. v2 < v1 -> new-dev was a mirage; debug.')
print('Bring the whole printout back.')

## Bring back to the session
The **verdict table**. This is the go/no-go for GRPO v2.